# 將CSV轉換為JSON檔案

將CSV資料轉換為符合 watsonx.ai 批次部署格式的JSON檔案

#### 處理流程
1. 使用Pandas分批讀取CSV（避免記憶體溢出）
2. 將每一筆資料轉換為JSON格式
3. 儲存為獨立的JSON檔案
4. 顯示進度條

## Import

In [2]:
import pandas as pd
import json
from pathlib import Path
import gc
import os
from datetime import datetime, timezone, timedelta
from ibm_watsonx_ai import APIClient
from ibm_watsonx_ai import Credentials
import getpass

# Set timezone to UTC+8 (Asia/Taipei)
TZ_UTC8 = timezone(timedelta(hours=8))

print(f"Pandas版本: {pd.__version__}")

Pandas版本: 2.2.3


## 設定 watsonx.ai 連線

In [3]:
# ============================================================================
# CONFIGURATION - Update with your credentials
# ============================================================================

CLOUD_API_KEY = getpass.getpass("Enter your IBM Cloud API key: ")
SPACE_ID = "165e735f-11f6-4879-a6fd-99cfecf44f37"

# 初始化 watsonx.ai client
credentials = Credentials(
    url="https://us-south.ml.cloud.ibm.com",
    api_key=CLOUD_API_KEY
)

client = APIClient(credentials)

# 設定 project 或 space
space = client.set.default_space(SPACE_ID)

print("✅ watsonx.ai client 初始化完成")

Enter your IBM Cloud API key:  ········


✅ watsonx.ai client 初始化完成


## 設定參數並下載 CSV

In [4]:
# ============================================================================
# CONFIGURATION - Update with your data asset ID
# ============================================================================

DATA_ASSET_ID = "33ff7a97-67ae-41bb-992b-41aabeabb4c3"  # 替換為你的 CSV data asset ID

# 配置參數
INPUT_DIR = Path("data/raw_csv")
INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR = Path("data/json_files")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"📥 從 watsonx.ai 下載 CSV 檔案...")
print(f"Data Asset ID: {DATA_ASSET_ID}")

# 從 watsonx.ai 下載 CSV 檔案
temp_csv_filename = INPUT_DIR / "downloaded_data.csv"
try:
    client.data_assets.download(DATA_ASSET_ID, str(temp_csv_filename))
    print(f"✅ CSV 檔案下載成功: {temp_csv_filename}")
except Exception as e:
    print(f"❌ 下載失敗: {e}")
    raise

INPUT_CSV = temp_csv_filename
print(f"輸入CSV: {INPUT_CSV}")
print(f"輸出目錄: {OUTPUT_DIR}")

# 分批處理參數
CHUNK_SIZE = 1000  # 每次處理1000筆

# 處理範圍設定
# 選項1: 處理全部資料
START_ROW = None  # 從第幾筆開始 (None = 從頭開始，1-based)
END_ROW = None    # 到第幾筆結束 (None = 處理到最後，1-based)

# 選項2: 處理指定範圍（取消註解並設定數值）
# START_ROW = 0      # 從第 n 筆開始
# END_ROW = 10000    # 處理到第 m 筆

print(f"\n分批大小: {CHUNK_SIZE:,} 筆/批次")
if START_ROW is not None or END_ROW is not None:
    print(f"處理範圍: 第 {START_ROW or 1} 筆 到 第 {END_ROW or '最後'} 筆")
else:
    print(f"處理範圍: 全部資料")

📥 從 watsonx.ai 下載 CSV 檔案...
Data Asset ID: 33ff7a97-67ae-41bb-992b-41aabeabb4c3
Successfully saved data asset content to file: 'data/raw_csv/downloaded_data.csv'
✅ CSV 檔案下載成功: data/raw_csv/downloaded_data.csv
輸入CSV: data/raw_csv/downloaded_data.csv
輸出目錄: data/json_files

分批大小: 1,000 筆/批次
處理範圍: 全部資料


## 定義轉換函數

In [5]:
def row_to_json_format(row_dict):
    """
    將單行資料轉換為watsonx.ai批次部署JSON格式
    
    完整的32個輸入結構：
    - 9個 MD_BRN_CONTACTS 輸入
    - 6個時間序列輸入
    - 9個 MD_CUS_RISK 輸入
    - 8個 MD_CUS_STYLE 輸入
    """
    input_data = []
    
    # 1. MD_BRN_CONTACTS - 連續特徵 (16個值)
    brn_continuous = [float(row_dict[f"input_x_o_MD_BRN_CONTACTS_{i:02d}"]) for i in range(16)]
    input_data.append({
        "id": "input_x_o_MD_BRN_CONTACTS",
        "values": [brn_continuous]
    })
    
    # 2. MD_BRN_CONTACTS - 類別特徵 (8個)
    for i in range(1, 9):
        input_data.append({
            "id": f"input_x_c_{i}_MD_BRN_CONTACTS",
            "values": [[int(row_dict[f"input_x_c_{i}_MD_BRN_CONTACTS"])]]
        })
    
    # 3. 時間序列特徵 - 前3個
    for ts_name in ["input_x_t_MD_CASH_FLOW", "input_x_t_MD_CUS_EXCH", "input_x_t_MD_CUS_LOAN"]:
        ts_data = json.loads(row_dict[ts_name])
        input_data.append({
            "id": ts_name,
            "values": [ts_data]
        })
    
    # 4. MD_CUS_RISK - 連續特徵 (1個值)
    input_data.append({
        "id": "input_x_o_MD_CUS_RISK",
        "values": [[float(row_dict["input_x_o_MD_CUS_RISK"])]]
    })
    
    # 5. MD_CUS_RISK - 類別特徵 (8個)
    for i in range(1, 9):
        input_data.append({
            "id": f"input_x_c_{i}_MD_CUS_RISK",
            "values": [[int(row_dict[f"input_x_c_{i}_MD_CUS_RISK"])]]
        })
    
    # 6. MD_CUS_STYLE - 連續特徵 (35個值)
    style_continuous = [float(row_dict[f"input_x_o_MD_CUS_STYLE_{i:02d}"]) for i in range(35)]
    input_data.append({
        "id": "input_x_o_MD_CUS_STYLE",
        "values": [style_continuous]
    })
    
    # 7. MD_CUS_STYLE - 類別特徵 (7個)
    for i in range(1, 8):
        input_data.append({
            "id": f"input_x_c_{i}_MD_CUS_STYLE",
            "values": [[int(row_dict[f"input_x_c_{i}_MD_CUS_STYLE"])]]
        })
    
    # 8. 時間序列特徵 - 後3個
    for ts_name in ["input_x_t_MD_DEP_AUM", "input_x_t_MD_NEW_INVST", "input_x_t_MD_PDH_AUM"]:
        ts_data = json.loads(row_dict[ts_name])
        input_data.append({
            "id": ts_name,
            "values": [ts_data]
        })
    
    return {"input_data": input_data}

print("轉換函數已定義")

轉換函數已定義


## 分批處理CSV並轉換為JSON

In [6]:
# 先計算總行數
print("計算總行數...")
total_rows = sum(1 for _ in open(INPUT_CSV)) - 1  # 減去標題行
print(f"CSV總共 {total_rows:,} 筆資料")

# 計算實際處理範圍
start_idx = (START_ROW - 1) if START_ROW is not None else 0
end_idx = (END_ROW - 1) if END_ROW is not None else (total_rows - 1)

# 驗證範圍
if start_idx < 0:
    start_idx = 0
if end_idx >= total_rows:
    end_idx = total_rows - 1
if start_idx > end_idx:
    raise ValueError(f"起始行 ({start_idx+1}) 不能大於結束行 ({end_idx+1})")

rows_to_process = end_idx - start_idx + 1
print(f"處理範圍: 第 {start_idx+1} 筆 到 第 {end_idx+1} 筆")
print(f"將處理: {rows_to_process:,} 筆資料\n")

# 分批讀取並轉換
record_count = 0
batch_count = 0
current_row = 0

print("開始轉換...")
for chunk in pd.read_csv(INPUT_CSV, chunksize=CHUNK_SIZE):
    batch_count += 1
    
    # 處理這個批次的每一筆資料
    for idx, row in chunk.iterrows():
        # 檢查是否在處理範圍內
        if current_row < start_idx:
            current_row += 1
            continue
        
        if current_row > end_idx:
            break
        
        # 轉換為字典
        row_dict = row.to_dict()
        
        # 轉換為JSON格式
        json_data = row_to_json_format(row_dict)
        
        # 儲存為JSON檔案（使用原始行號作為檔名）
        json_filename = OUTPUT_DIR / f"record_{current_row:07d}.json"
        with open(json_filename, 'w', encoding='utf-8') as f:
            json.dump(json_data, f, ensure_ascii=False)
        
        record_count += 1
        current_row += 1
        
        # 每1000筆顯示一次進度
        if record_count % 1000 == 0:
            print(f"已處理: {record_count:,} / {rows_to_process:,} 筆 ({record_count/rows_to_process*100:.1f}%)")
    
    # 如果已經超過結束行，提前結束
    if current_row > end_idx:
        break
    
    # 每處理10個批次清理一次記憶體
    if batch_count % 10 == 0:
        gc.collect()

print(f"\n完成！")
print(f"總共處理: {record_count:,} 筆資料")
print(f"處理範圍: 第 {start_idx+1} 筆 到 第 {start_idx+record_count} 筆")
print(f"總共批次: {batch_count} 批")

計算總行數...
CSV總共 1,000 筆資料
處理範圍: 第 1 筆 到 第 1000 筆
將處理: 1,000 筆資料

開始轉換...
已處理: 1,000 / 1,000 筆 (100.0%)

完成！
總共處理: 1,000 筆資料
處理範圍: 第 1 筆 到 第 1000 筆
總共批次: 1 批


## 驗證輸出

In [7]:
# 統計JSON檔案數量
json_files = list(OUTPUT_DIR.glob("record_*.json"))
print(f"JSON檔案數量: {len(json_files):,}")

# 計算總大小
total_size = sum(f.stat().st_size for f in json_files)
print(f"總大小: {total_size / (1024**2):.2f} MB")
print(f"平均每個檔案: {total_size / len(json_files) / 1024:.2f} KB")

JSON檔案數量: 1,000
總大小: 31.32 MB
平均每個檔案: 32.07 KB


In [8]:
# 驗證第一個JSON檔案的結構
first_json = OUTPUT_DIR / "record_0000000.json"
with open(first_json, 'r', encoding='utf-8') as f:
    sample_data = json.load(f)

print(f"\n第一個JSON檔案結構驗證:")
print(f"總輸入數: {len(sample_data['input_data'])}")
print(f"\n所有輸入的形狀:")

for i, item in enumerate(sample_data['input_data'], 1):
    values = item['values'][0]
    if isinstance(values, list) and len(values) > 0 and isinstance(values[0], list):
        shape = f"({len(values)}, {len(values[0])})"
    elif isinstance(values, list):
        shape = f"({len(values)})"
    else:
        shape = "(1)"
    print(f"{i:2d}. {item['id']}: {shape}")

print(f"\n✅ 結構驗證完成！")


第一個JSON檔案結構驗證:
總輸入數: 32

所有輸入的形狀:
 1. input_x_o_MD_BRN_CONTACTS: (16)
 2. input_x_c_1_MD_BRN_CONTACTS: (1)
 3. input_x_c_2_MD_BRN_CONTACTS: (1)
 4. input_x_c_3_MD_BRN_CONTACTS: (1)
 5. input_x_c_4_MD_BRN_CONTACTS: (1)
 6. input_x_c_5_MD_BRN_CONTACTS: (1)
 7. input_x_c_6_MD_BRN_CONTACTS: (1)
 8. input_x_c_7_MD_BRN_CONTACTS: (1)
 9. input_x_c_8_MD_BRN_CONTACTS: (1)
10. input_x_t_MD_CASH_FLOW: (12, 22)
11. input_x_t_MD_CUS_EXCH: (12, 12)
12. input_x_t_MD_CUS_LOAN: (12, 24)
13. input_x_o_MD_CUS_RISK: (1)
14. input_x_c_1_MD_CUS_RISK: (1)
15. input_x_c_2_MD_CUS_RISK: (1)
16. input_x_c_3_MD_CUS_RISK: (1)
17. input_x_c_4_MD_CUS_RISK: (1)
18. input_x_c_5_MD_CUS_RISK: (1)
19. input_x_c_6_MD_CUS_RISK: (1)
20. input_x_c_7_MD_CUS_RISK: (1)
21. input_x_c_8_MD_CUS_RISK: (1)
22. input_x_o_MD_CUS_STYLE: (35)
23. input_x_c_1_MD_CUS_STYLE: (1)
24. input_x_c_2_MD_CUS_STYLE: (1)
25. input_x_c_3_MD_CUS_STYLE: (1)
26. input_x_c_4_MD_CUS_STYLE: (1)
27. input_x_c_5_MD_CUS_STYLE: (1)
28. input_x_c_6_MD_CUS_STY

## 與參考JSON比對 (optional)

In [ ]:
# # 讀取參考JSON
# reference_json = Path("./data/single_batch_savedmodel_scoring_input_data_example.json")

# if reference_json.exists():
#     with open(reference_json, 'r', encoding='utf-8') as f:
#         ref_data = json.load(f)
    
#     print("與參考JSON比對:")
#     print(f"\n參考JSON輸入數: {len(ref_data['input_data'])}")
#     print(f"生成JSON輸入數: {len(sample_data['input_data'])}")
    
#     # 比對每個輸入的ID和形狀
#     print(f"\n詳細比對:")
#     all_match = True
#     for i, (ref_item, gen_item) in enumerate(zip(ref_data['input_data'], sample_data['input_data']), 1):
#         ref_id = ref_item['id']
#         gen_id = gen_item['id']
        
#         ref_values = ref_item['values'][0]
#         gen_values = gen_item['values'][0]
        
#         # 計算形狀
#         if isinstance(ref_values, list) and len(ref_values) > 0 and isinstance(ref_values[0], list):
#             ref_shape = f"({len(ref_values)}, {len(ref_values[0])})"
#             gen_shape = f"({len(gen_values)}, {len(gen_values[0])})"
#         elif isinstance(ref_values, list):
#             ref_shape = f"({len(ref_values)})"
#             gen_shape = f"({len(gen_values)})"
#         else:
#             ref_shape = "(1)"
#             gen_shape = "(1)"
        
#         match = (ref_id == gen_id) and (ref_shape == gen_shape)
#         status = "✅" if match else "❌"
        
#         if not match:
#             all_match = False
#             print(f"{i:2d}. {status} 參考: {ref_id} {ref_shape} | 生成: {gen_id} {gen_shape}")
    
#     if all_match:
#         print("\n🎉 完美匹配！所有輸入的ID和形狀都與參考JSON一致！")
#     else:
#         print("\n⚠️ 發現不匹配的項目，請檢查上方詳細資訊")
# else:
#     print(f"⚠️ 找不到參考JSON: {reference_json}")

## 總結

已成功將CSV轉換為JSON檔案：
- 每筆資料一個獨立的JSON檔案
- 完整的32個輸入結構
- 符合watsonx.ai批次部署格式

下一步：使用 `2_zip_all_json_files.ipynb` 將所有JSON檔案壓縮為單一ZIP檔